# 🧪 Session 11 exercises
### Trees and forests

The lecture built a tree from its parts: the error of a box, the best cut, one question at a time, and a formula for the boxes at the end. Then it grew trees too deep, averaged many of them, and tuned a forest on the time-series folds. These exercises do each of those steps yourself, with loops and functions first and scikit-learn second.

Two tables, the same two as the lecture. The **index table** from Session 6, with the volatility over the next 20 days as the target. The **credit table** from Session 9, with a default as the label.

## How to use this notebook

- Run the **setup cell** below first. It loads both tables and the eleven instruments, makes the splits, and imports the scikit-learn pieces.
- Each exercise has a **task**, then a **code cell** for your work. Cells with `...` are blanks to fill in. Replace them with real code.
- Stuck? Open the **💡 Hint**, but only after a genuine attempt. Open the **✅ Solution** to *check* yourself, not to skip the thinking.
- Every cell runs cleanly even with the blanks still in place, so pressing **Run all** never floods you with errors.
- Most exercises stand alone. A few short runs build on each other (A3 to A4, B2 to B3, D3 to D4); the task says which earlier exercise it continues from. Section I shows five mistakes and asks you to fix them. Section J uses the function from J1, and **section K is five small cases that each start from scratch**. Ten of them ask you to draw something: A5, C1, C5, D2, E3, F3, G3, H3, H6 and J5.
- A few cells fit forests or search grids and take several seconds. The task says so where it matters.

**You are not expected to finish all of these.** Do what you can, and come back to the rest when you revise. Short on time? Read the hint, then the solution. A worked solution you genuinely understand is real learning too.

**Units.** The index table is in percent, as in the lecture. The credit table is in New Taiwan dollars, as it comes.

### Difficulty

| badge | what to expect |
|:--|:--|
| ★☆☆☆☆ | One step, straight from the lecture. You are checking that you can type it. |
| ★★☆☆☆ | The same idea on new data, or two steps in a row. Nothing to decide. |
| ★★★☆☆ | Combine two ideas, or adapt a pattern rather than copy it. |
| ★★★★☆ | You choose the approach. Several steps, and something has to be worked out before you type. |
| ★★★★★ | A genuine puzzle: an insight, or a constraint that rules out the obvious route. Always solvable with what you have. |

The stars rate the work against **this** session. A three-star task here assumes everything from Sessions 1 to 10, so it is a bigger piece of work than a three-star task in an earlier notebook.

Some exercises also carry a **revisits** tag. Those need something from an earlier session as well as today's material, and they are there on purpose: the skills are meant to accumulate.

---

## 🧰 Toolkit

New this session. Hover a name for what it does.

<span title="A tree for a number. Each leaf forecasts the mean of its training rows. max_depth, min_samples_leaf and ccp_alpha limit its size.">`DecisionTreeRegressor`</span> · <span title="A tree for a label. Each leaf holds the share of each class, and predict_proba returns it.">`DecisionTreeClassifier`</span> · <span title="Draws a fitted tree with matplotlib. feature_names names the columns; filled shades the boxes; impurity=False drops the error line; label=root names the fields in the top box only; precision and fontsize set the text.">`plot_tree`</span> · <span title="The number of questions on the longest path, and the number of boxes at the end.">`.get_depth()` `.get_n_leaves()`</span> · <span title="Inside a fitted tree: tree_.feature[0] is the column number of the first question and tree_.threshold[0] its cut.">`tree_`</span> · <span title="Grows each tree on a bootstrap sample and averages them. The trees are kept in estimators_.">`BaggingRegressor`</span> · <span title="Bagging where each question chooses among max_features columns drawn at random. oob_score=True scores each row with the trees that did not see it.">`RandomForestRegressor`</span> · <span title="The same for a label. oob_decision_function_ holds the out-of-bag probabilities.">`RandomForestClassifier`</span> · <span title="Reads a clock in seconds. The difference of two readings is the time between them.">`time.perf_counter`</span> · <span title="Tries n_iter combinations drawn at random from a grid, instead of all of them.">`RandomizedSearchCV`</span> · <span title="n_jobs=-1 fits the trees of a forest, or the folds of a search, on every core.">`n_jobs`</span>

**Formulas**

- the forecast of a box = the mean of its training rows
- the error of a box: RSS = sum of (y minus the box mean) squared; for labels of 0 and 1, k(n minus k)/n with k ones among n rows
- a cut is chosen to make RSS(left) + RSS(right) as small as possible
- Gini for two classes = 2p(1 minus p), with p the share of ones
- the share of distinct rows in a bootstrap sample of n rows is about 1 minus (1 minus 1/n) to the power n, which is about 0.632
- combinations in a grid = the product of the lengths of its lists; fits = combinations x folds

---

## ⚙️ Setup · run me first

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, BaggingRegressor
from sklearn.metrics import mean_squared_error, roc_auc_score
from sklearn.model_selection import (train_test_split, cross_val_score, TimeSeriesSplit,
                                     GridSearchCV, RandomizedSearchCV)

CANDIDATE_DIRS = ["data", os.path.join("..", "data"), "."]
REPO_RAW_URL = "https://raw.githubusercontent.com/theill95/mlfin-2026/main/data/"   # used when the CSV files are not next to the notebook


def data_path(filename):
    """Where the course CSV files are, wherever you happen to be running."""
    for folder in CANDIDATE_DIRS:
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            return path
    if REPO_RAW_URL is not None:
        return REPO_RAW_URL + filename
    raise FileNotFoundError(
        f"Could not find {filename}. Run this notebook from the course folder, "
        f"upload the CSV into Colab, or set REPO_RAW_URL."
    )


def rmse(actual, predicted):
    """Root mean squared error, as in Sessions 5 and 6."""
    return np.sqrt(mean_squared_error(actual, predicted))


# ---- the index table from Session 6: one row per trading day, percent ----
table = pd.read_csv(data_path("market_features.csv"), parse_dates=["date"]).set_index("date")
columns = list(table.columns[:-1])      # the 19 feature columns
pair = ["vol_20d", "ret_20d"]
train = table.loc[:"2022-12-31"]
test = table.loc["2023-01-01":]
folds = TimeSeriesSplit(n_splits=5)

# ---- the credit table from Session 9: one row per borrower, no time order ----
credit = pd.read_csv(data_path("credit.csv"))
c_columns = ["limit", "age", "late_now", "months_late", "bill", "paid", "utilisation"]
c_train, c_test = train_test_split(credit, test_size=0.3, random_state=0,
                                   stratify=credit["default"])

# ---- the eleven instruments, for section J. Returns in percent. ----
prices = pd.read_csv(data_path("prices.csv"), parse_dates=["date"])
tickers = sorted(prices["ticker"].unique())
rets = prices.pivot(index="date", columns="ticker", values="close").pct_change().dropna() * 100

print("index :", table.shape, "  train", len(train), " test", len(test))
print("credit:", credit.shape, "  default share", round(credit["default"].mean(), 4))
print("prices:", prices.shape[0], "rows,", len(tickers), "instruments")

---

## A · A box and its error

A tree forecasts with boxes, and every box forecasts one number. These exercises find that number and the error it leaves, with NumPy and loops first. A3 and A4 run together.

### A1 · The forecast of one box  ★☆☆☆☆  · revisits S3

The six days of the lecture are below. A box forecasts the mean of its days. Compute the mean of `vol_next` and the box's RSS, the sum of squared distances from that mean, with NumPy.

In [ ]:
vol_next = np.array([0.5, 0.6, 0.7, 1.2, 1.5, 1.2])

mean = ...
box_rss = ...

print(mean)
print(box_rss)

<details>
<summary>💡 Hint</summary>

`vol_next.mean()` is the mean. Subtract it from the array, square, and `.sum()`. Wrap each in `round(float(...), 3)` for a plain number.

</details>

<details>
<summary>✅ Solution</summary>

```python
vol_next = np.array([0.5, 0.6, 0.7, 1.2, 1.5, 1.2])

mean = round(float(vol_next.mean()), 3)
box_rss = round(float(((vol_next - vol_next.mean()) ** 2).sum()), 3)

print(mean)
print(box_rss)
```

The box forecasts 0.95 for all six days and leaves an RSS of 0.815. Subtracting a number from an array subtracts it from every element, which is what makes this one line instead of a loop.

</details>

---

### A2 · No constant beats the mean  ★★☆☆☆  · revisits S3

Try every constant from 0.5 to 1.5 in steps of 0.01 as the forecast for the same six days. Compute the RSS for each, and print the constant with the smallest RSS.

In [ ]:
vol_next = np.array([0.5, 0.6, 0.7, 1.2, 1.5, 1.2])
constants = np.arange(0.5, 1.5, 0.01)
totals = []
for c in constants:
    ...

best = ...

print(best)

<details>
<summary>💡 Hint 1</summary>

Inside the loop: `totals.append(((vol_next - c) ** 2).sum())`.

</details>

<details>
<summary>💡 Hint 2</summary>

`np.argmin(totals)` is the POSITION of the smallest total, so the constant is `constants[np.argmin(totals)]`.

</details>

<details>
<summary>✅ Solution</summary>

```python
vol_next = np.array([0.5, 0.6, 0.7, 1.2, 1.5, 1.2])
constants = np.arange(0.5, 1.5, 0.01)
totals = []
for c in constants:
    totals.append(((vol_next - c) ** 2).sum())

best = round(float(constants[np.argmin(totals)]), 2)

print(best)
```

0.95, the mean from A1. The lecture proved it with a derivative; here a hundred tries find the same answer. The mean is the best single number under squared error, which is why every leaf of a regression tree forecasts a mean.

</details>

---

### A3 · The error of a box, as a function  ★★☆☆☆  · revisits S2

Write `rss(values)` returning the sum of squared distances from the mean of `values`, rounded to three decimals, with a docstring. It should work on a list, a NumPy array and a pandas Series. Test it on the six days and on the `vol_next` of all training days.

In [ ]:
def rss(values):
    """..."""
    ...


print(rss([0.5, 0.6, 0.7, 1.2, 1.5, 1.2]))
print(rss(train['vol_next']))

<details>
<summary>💡 Hint</summary>

`values = np.asarray(values, dtype=float)` turns any of the three into an array first. Then the same line as A1.

</details>

<details>
<summary>✅ Solution</summary>

```python
def rss(values):
    """The sum of squared distances from the mean: the error of a box."""
    values = np.asarray(values, dtype=float)
    return round(float(((values - values.mean()) ** 2).sum()), 3)


print(rss([0.5, 0.6, 0.7, 1.2, 1.5, 1.2]))
print(rss(train['vol_next']))
```

0.815 and 908.526. The second is the error of a tree with no question at all: every training day forecast by the one mean. Keep `rss`: A4 continues from here.

</details>

---

### A4 · Every cut on the six  ★★☆☆☆  · revisits S2

Using `rss` from A3, loop over the cuts after 0.4, 0.6, 0.8, 1.2 and 1.6 on the six days. Store each cut's total RSS, left box plus right box, in a dictionary, and print the dictionary and the cut with the smallest total.

In [ ]:
six = pd.DataFrame({'vol_20d': [0.4, 0.6, 0.8, 1.2, 1.6, 2.4],
                    'vol_next': [0.5, 0.6, 0.7, 1.2, 1.5, 1.2]})
by_cut = {}
for cut in [0.4, 0.6, 0.8, 1.2, 1.6]:
    ...

best_cut = ...

print(by_cut)
print(best_cut)

<details>
<summary>💡 Hint 1</summary>

The left box is `six[six['vol_20d'] <= cut]['vol_next']`, the right box the same with `>`.

</details>

<details>
<summary>💡 Hint 2</summary>

`min(by_cut, key=by_cut.get)` gives the KEY with the smallest value.

</details>

<details>
<summary>✅ Solution</summary>

```python
six = pd.DataFrame({'vol_20d': [0.4, 0.6, 0.8, 1.2, 1.6, 2.4],
                    'vol_next': [0.5, 0.6, 0.7, 1.2, 1.5, 1.2]})
by_cut = {}
for cut in [0.4, 0.6, 0.8, 1.2, 1.6]:
    left = six[six['vol_20d'] <= cut]['vol_next']
    right = six[six['vol_20d'] > cut]['vol_next']
    by_cut[cut] = round(rss(left) + rss(right), 3)

best_cut = min(by_cut, key=by_cut.get)

print(by_cut)
print(best_cut)
```

`{0.4: 0.572, 0.6: 0.335, 0.8: 0.08, 1.2: 0.335, 1.6: 0.74}`, and the cut after 0.8 wins with 0.08. It puts the three calm days in one box and the three busy days in the other, and each box is then close to its own mean.

</details>

---

### A5 · Draw the RSS of every cut  ★★☆☆☆  · revisits S3

For 60 cuts on `vol_20d`, the percentiles from 2 to 98 of the training days, compute the total RSS each cut leaves on `vol_next`. Draw the total against the cut and mark the lowest point.

In [ ]:
cuts = np.quantile(train['vol_20d'], np.linspace(0.02, 0.98, 60))
totals = []
for cut in cuts:
    left = train[train['vol_20d'] <= cut]['vol_next']
    right = train[train['vol_20d'] > cut]['vol_next']
    ...

fig, ax = plt.subplots(figsize=(8, 3))
...
...
ax.set_xlabel('cut on vol_20d')
ax.set_ylabel('total RSS')
plt.show()

<details>
<summary>💡 Hint 1</summary>

Inside the loop: `totals.append(((left - left.mean()) ** 2).sum() + ((right - right.mean()) ** 2).sum())`.

</details>

<details>
<summary>💡 Hint 2</summary>

`ax.plot(cuts, totals)` for the curve. The lowest point is at `cuts[np.argmin(totals)]` and `min(totals)`; mark it with `ax.scatter`.

</details>

<details>
<summary>✅ Solution</summary>

```python
cuts = np.quantile(train['vol_20d'], np.linspace(0.02, 0.98, 60))
totals = []
for cut in cuts:
    left = train[train['vol_20d'] <= cut]['vol_next']
    right = train[train['vol_20d'] > cut]['vol_next']
    totals.append(((left - left.mean()) ** 2).sum() + ((right - right.mean()) ** 2).sum())

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(cuts, totals, color='#1c5cab')
ax.scatter([cuts[np.argmin(totals)]], [min(totals)], color='#b3402f', zorder=3)
ax.set_xlabel('cut on vol_20d')
ax.set_ylabel('total RSS')
ax.set_title('Total RSS left by each cut, training days', loc='left')
plt.show()
```

With no cut the total is 908.5. On this grid the curve is lowest at a cut of 1.122, with 743.0, and within 5 of that from 0.95 to 1.12. The exact best cut, 0.986 with 742.2, sits in the same flat stretch, so a tree grown on slightly different days can cut anywhere along it.

</details>

---

---

## B · The first question

A cut is a mask, and the best cut is the one that leaves the smallest RSS. These search for it with loops, then check the answer against scikit-learn. B2 and B3 run together.

### B1 · A cut as a mask  ★☆☆☆☆  · revisits S3

Split the training days at a `vol_20d` of 1.0. Count the days on each side and print the mean `vol_next` of each side.

In [ ]:
calm = ...
busy = ...
counts = ...
means = ...

print(counts)
print(means)

<details>
<summary>💡 Hint</summary>

`train[train['vol_20d'] <= 1.0]` keeps the calm side. Store the counts as a tuple `(len(calm), len(busy))`.

</details>

<details>
<summary>✅ Solution</summary>

```python
calm = train[train['vol_20d'] <= 1.0]
busy = train[train['vol_20d'] > 1.0]
counts = (len(calm), len(busy))
means = (round(float(calm['vol_next'].mean()), 3), round(float(busy['vol_next'].mean()), 3))

print(counts)
print(means)
```

1,163 calm days forecast 0.749 and 731 busy days forecast 1.354. That is a stump written by hand: one question, two boxes, two means.

</details>

---

### B2 · The best cut, by a loop  ★★★★☆  · revisits S2

Write `best_split(rows, col)` that tries every distinct value of `rows[col]` as a cut, except the largest, and returns the cut with the smallest total RSS of `vol_next` and that total. Run it on the training days with `'vol_20d'`. It takes a second or two.

In [ ]:
def best_split(rows, col):
    """..."""
    ...


print(best_split(train, 'vol_20d'))

<details>
<summary>💡 Hint 1</summary>

`np.unique(rows[col])[:-1]` gives the sorted distinct values without the largest. Loop over them, build the two boxes of `rows['vol_next']` with masks, and add up their RSS.

</details>

<details>
<summary>💡 Hint 2</summary>

Keep the best so far in two variables that start as `None` and `np.inf`, and replace them with an `if total < best_total:`.

</details>

<details>
<summary>✅ Solution</summary>

```python
def best_split(rows, col):
    """The cut on col with the smallest total RSS of vol_next, and that total."""
    y = rows['vol_next']
    best, best_total = None, np.inf
    for cut in np.unique(rows[col])[:-1]:
        left, right = y[rows[col] <= cut], y[rows[col] > cut]
        total = ((left - left.mean()) ** 2).sum() + ((right - right.mean()) ** 2).sum()
        if total < best_total:
            best, best_total = cut, total
    return round(float(best), 4), round(float(best_total), 1)


print(best_split(train, 'vol_20d'))
```

`(0.9856, 742.2)`: the best cut leaves 742.2, down from 908.5 with no cut. scikit-learn tries every value too. Keeping the best so far in a variable, and replacing it when something beats it, is the pattern behind every search by loop.

</details>

---

### B3 · The same cut from a stump  ★★☆☆☆

Continuing from B2. Fit a `DecisionTreeRegressor` with `max_depth=1` on `vol_20d` alone. Its cut is `stump.tree_.threshold[0]`. Print it beside the cut from B2, and explain the difference in a comment.

In [ ]:
stump = ...
...
cut = ...

print(cut)
print(best_split(train, 'vol_20d'))

# why do the two cuts differ?

<details>
<summary>💡 Hint</summary>

Fit on `train[['vol_20d']]` and `train['vol_next']`, then `round(float(stump.tree_.threshold[0]), 4)`.

</details>

<details>
<summary>✅ Solution</summary>

```python
stump = DecisionTreeRegressor(max_depth=1, random_state=0)
stump.fit(train[['vol_20d']], train['vol_next'])
cut = round(float(stump.tree_.threshold[0]), 4)

print(cut)
print(best_split(train, 'vol_20d'))

# best_split cuts AT a training value; scikit-learn cuts halfway between the
# last value on the left and the first on the right. The boxes are the same.
```

0.9859 against 0.9856. The next training value above 0.9856 is 0.9863, and scikit-learn puts its cut halfway between the two. Every training day lands in the same box either way, so the RSS is the same.

</details>

---

### B4 · Which column asks first  ★★★☆☆  · revisits S2

Fit a stump on each of the 19 columns alone and record its training RMSE in a dictionary. Which column makes the best first question? Then fit one stump on all 19 columns and check it picks the same one: its column number is `tree_.feature[0]`.

In [ ]:
errors = {}
for col in columns:
    ...

best_col = ...
together = ...
...
first = ...

print(best_col)
print(first)

<details>
<summary>💡 Hint 1</summary>

Inside the loop, fit `DecisionTreeRegressor(max_depth=1, random_state=0)` on `train[[col]]` and store `rmse(train['vol_next'], s.predict(train[[col]]))`.

</details>

<details>
<summary>💡 Hint 2</summary>

`first = columns[together.tree_.feature[0]]`: the number indexes the list of names.

</details>

<details>
<summary>✅ Solution</summary>

```python
errors = {}
for col in columns:
    s = DecisionTreeRegressor(max_depth=1, random_state=0).fit(train[[col]], train['vol_next'])
    errors[col] = round(float(rmse(train['vol_next'], s.predict(train[[col]]))), 4)

best_col = min(errors, key=errors.get)
together = DecisionTreeRegressor(max_depth=1, random_state=0)
together.fit(train[columns], train['vol_next'])
first = columns[together.tree_.feature[0]]

print(best_col)
print(first)
```

Both are `vol_5d`, with `vol_10d` second. The smallest training RMSE is the smallest RSS, because every stump is scored on the same days, so the loop and scikit-learn use one criterion. The stump on all 19 cuts at 3.251 and sends 27 days to the busy side: the first question sets aside the most volatile weeks.

</details>

---

### B5 · One question inside each box  ★★★☆☆

Split the training days at 0.986 on `vol_20d`. Fit a stump on `pair` inside each half, and store in a dictionary which column each stump asks about and where it cuts.

In [ ]:
calm = train[train['vol_20d'] <= 0.986]
busy = train[train['vol_20d'] > 0.986]
questions = {}
for name, rows in [('calm', calm), ('busy', busy)]:
    ...

print(questions)

<details>
<summary>💡 Hint</summary>

Inside the loop: fit the stump on `rows[pair]` and `rows['vol_next']`, then `questions[name] = (pair[s.tree_.feature[0]], round(float(s.tree_.threshold[0]), 3))`.

</details>

<details>
<summary>✅ Solution</summary>

```python
calm = train[train['vol_20d'] <= 0.986]
busy = train[train['vol_20d'] > 0.986]
questions = {}
for name, rows in [('calm', calm), ('busy', busy)]:
    s = DecisionTreeRegressor(max_depth=1, random_state=0).fit(rows[pair], rows['vol_next'])
    questions[name] = (pair[s.tree_.feature[0]], round(float(s.tree_.threshold[0]), 3))

print(questions)
```

`{'calm': ('vol_20d', 0.823), 'busy': ('ret_20d', -0.453)}`. The calm box asks about the level again, the busy box about the direction. Two stumps grown in two boxes are exactly a tree of depth 2, and the different second questions are the interaction.

</details>

---

---

## C · Fitting, drawing and reading trees

`plot_tree` shows a fitted tree, and its forecasts can be counted and traced. Every exercise here fits its own tree.

### C1 · Draw a tree of depth 2  ★☆☆☆☆

Fit a tree of depth 2 on `pair` and draw it with `plot_tree`, with the column names and filled boxes.

In [ ]:
small = ...
...

plt.figure(figsize=(10, 3))
...
plt.show()

<details>
<summary>💡 Hint</summary>

`plot_tree(small, feature_names=pair, filled=True, fontsize=10)`.

</details>

<details>
<summary>✅ Solution</summary>

```python
small = DecisionTreeRegressor(max_depth=2, random_state=0)
small.fit(train[pair], train['vol_next'])

plt.figure(figsize=(10, 3))
plot_tree(small, feature_names=pair, filled=True, fontsize=10)
plt.show()
```

Four leaves: 0.698 for 969 days, 0.994 for 185, 2.84 for 47 and 1.252 for 693. The darkest box is the small one where volatility was high after a fall.

</details>

---

### C2 · The forecasts a tree can give  ★★☆☆☆  · revisits S3

Fit a tree of depth 2 on `pair`, forecast the test days, and count how many test days receive each distinct forecast.

In [ ]:
small = ...
...

forecasts = ...
counts = ...

print(counts)

<details>
<summary>💡 Hint</summary>

`pd.Series(small.predict(test[pair]), index=test.index).round(3)`, then `.value_counts()`.

</details>

<details>
<summary>✅ Solution</summary>

```python
small = DecisionTreeRegressor(max_depth=2, random_state=0)
small.fit(train[pair], train['vol_next'])

forecasts = pd.Series(small.predict(test[pair]), index=test.index).round(3)
counts = forecasts.value_counts()

print(counts)
```

Three forecasts for 482 days: 293 days get 0.698, 102 get 0.994 and 87 get 1.252. The fourth leaf is never used, because no test day was volatile after a steep fall.

</details>

---

### C3 · Depth and leaves  ★☆☆☆☆

For depths 1 to 6, fit a tree on `pair` and store its number of leaves in a dictionary keyed by the depth.

In [ ]:
leaves = {}
for depth in range(1, 7):
    ...

print(leaves)

<details>
<summary>💡 Hint</summary>

Inside the loop, fit the tree and store `t.get_n_leaves()`.

</details>

<details>
<summary>✅ Solution</summary>

```python
leaves = {}
for depth in range(1, 7):
    t = DecisionTreeRegressor(max_depth=depth, random_state=0).fit(train[pair], train['vol_next'])
    leaves[depth] = int(t.get_n_leaves())

print(leaves)
```

`{1: 2, 2: 4, 3: 8, 4: 16, 5: 31, 6: 60}`. Each level can at most double the leaves, so depth d allows 2 to the power d. At depth 6 there are 60 rather than 64, because some boxes were already down to a single day.

</details>

---

### C4 · One day through the tree  ★★☆☆☆  · revisits S3

Fit a tree of depth 2 on `pair`. Make a one-row DataFrame for a day with `vol_20d` of 1.5 and `ret_20d` of -0.6, and forecast it. Then do the same with a `ret_20d` of 0.1.

In [ ]:
small = ...
...

day = ...
other = ...

print(...)
print(...)

<details>
<summary>💡 Hint</summary>

`pd.DataFrame({'vol_20d': [1.5], 'ret_20d': [-0.6]})`. The columns must have the names the tree was fitted on.

</details>

<details>
<summary>✅ Solution</summary>

```python
small = DecisionTreeRegressor(max_depth=2, random_state=0)
small.fit(train[pair], train['vol_next'])

day = pd.DataFrame({'vol_20d': [1.5], 'ret_20d': [-0.6]})
other = pd.DataFrame({'vol_20d': [1.5], 'ret_20d': [0.1]})

print(small.predict(day))
print(small.predict(other))
```

2.84 and 1.252: the same level of volatility gets a forecast more than twice as high after a steep fall. Linear regression on the same two columns gives 1.585 and 1.187. It also forecasts more after a fall, but by the same amount at every level of volatility; the tree asks about the fall only in busy weeks.

</details>

---

### C5 · A deeper tree, drawn so it fits  ★★☆☆☆

Fit a tree of depth 3 on all 19 columns, and draw it so that its eight leaves fit on one screen: field names in the top box only, two decimals, no error line, and small text.

In [ ]:
tree = ...
...

plt.figure(figsize=(14, 4))
...
plt.show()

<details>
<summary>💡 Hint</summary>

`plot_tree(tree, feature_names=columns, filled=True, impurity=False, label='root', precision=2, fontsize=8)`.

</details>

<details>
<summary>✅ Solution</summary>

```python
tree = DecisionTreeRegressor(max_depth=3, random_state=0)
tree.fit(train[columns], train['vol_next'])

plt.figure(figsize=(14, 4))
plot_tree(tree, feature_names=columns, filled=True, impurity=False,
          label='root', precision=2, fontsize=8)
plt.show()
```

The first question is about `vol_5d`, and both second questions about `DIS_vol`, Disney's volatility rather than the index's own. Two of the eight leaves hold 1 and 2 days. A tree this size spends its questions on a handful of extreme days, and it scores 0.390 on the test days.

</details>

---

### C6 · Which leaf each day lands in  ★★★☆☆  · revisits S3

`.apply(X)` returns, for every row of `X`, the number of the leaf it lands in. Fit a tree of depth 3 on all 19 columns, apply it to the test days, count the days in each leaf, and print the share of test days in the busiest leaf.

In [ ]:
tree = ...
...

leaf_ids = ...
per_leaf = ...
share_busiest = ...

print(per_leaf)
print(share_busiest)

<details>
<summary>💡 Hint</summary>

`pd.Series(tree.apply(test[columns])).value_counts()`, then its `.max()` divided by `len(test)`.

</details>

<details>
<summary>✅ Solution</summary>

```python
tree = DecisionTreeRegressor(max_depth=3, random_state=0)
tree.fit(train[columns], train['vol_next'])

leaf_ids = tree.apply(test[columns])
per_leaf = pd.Series(leaf_ids).value_counts()
share_busiest = round(per_leaf.max() / len(test), 3)

print(per_leaf)
print(share_busiest)
```

The 482 test days use only 2 of the eight leaves, and 330 of them, 68.5 percent, land in one. Those days all get 1.197, while their volatility over the next 20 days averaged 0.80. A leaf learned in busier years forecasts a busy month for calm ones.

</details>

---

---

## D · Depth and overfitting

Deeper trees fit the training days better and the test days worse. D3 and D4 run together.

### D1 · A tree without a limit  ★☆☆☆☆

Fit a tree with no depth limit on `pair`. Print its depth and number of leaves, then its training and test RMSE.

In [ ]:
full = ...
...
size = ...
errors = ...

print(size)
print(errors)

<details>
<summary>💡 Hint</summary>

`size = (full.get_depth(), int(full.get_n_leaves()))` and the errors from `rmse` on the training and the test days.

</details>

<details>
<summary>✅ Solution</summary>

```python
full = DecisionTreeRegressor(random_state=0)
full.fit(train[pair], train['vol_next'])
size = (full.get_depth(), int(full.get_n_leaves()))
errors = (round(float(rmse(train['vol_next'], full.predict(train[pair]))), 4),
          round(float(rmse(test['vol_next'], full.predict(test[pair]))), 4))

print(size)
print(errors)
```

Depth 32, 1,894 leaves, a training RMSE of 0.0 and a test RMSE of 0.4663. One leaf per training day reproduces every training day and forecasts the test days from single past days.

</details>

---

### D2 · Training and test error against depth  ★★☆☆☆  · revisits S3

For depths 1 to 15, fit a tree on `pair`, record the training and the test RMSE, and draw both against the depth with a legend.

In [ ]:
depths = range(1, 16)
train_err = []
test_err = []
for depth in depths:
    ...

fig, ax = plt.subplots(figsize=(8, 3))
...
...
...
ax.set_xlabel('max_depth')
ax.set_ylabel('RMSE')
plt.show()

<details>
<summary>💡 Hint 1</summary>

Inside the loop: fit, then append `rmse(train['vol_next'], t.predict(train[pair]))` and the same on the test days.

</details>

<details>
<summary>💡 Hint 2</summary>

`ax.plot(depths, train_err, label='training days')` and the same for the test days, so `ax.legend()` has names to show.

</details>

<details>
<summary>✅ Solution</summary>

```python
depths = range(1, 16)
train_err = []
test_err = []
for depth in depths:
    t = DecisionTreeRegressor(max_depth=depth, random_state=0).fit(train[pair], train['vol_next'])
    train_err.append(rmse(train['vol_next'], t.predict(train[pair])))
    test_err.append(rmse(test['vol_next'], t.predict(test[pair])))

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(depths, train_err, marker='o', label='training days')
ax.plot(depths, test_err, marker='o', label='test days')
ax.set_xlabel('max_depth')
ax.set_ylabel('RMSE')
ax.legend()
plt.show()
```

The training error falls at every step, from 0.626 to 0.130. The test error is lowest at depths 1 and 2, 0.269, and above 0.43 from depth 10 on. The gap between the two lines is the variance a deep tree carries.

</details>

---

### D3 · Depth on the time-series folds  ★★☆☆☆  · revisits S6

Search `max_depth` over 1, 2, 3, 4, 6 and 8 with `GridSearchCV`, `folds` from the setup cell, and RMSE as the score. Print the best depth, its mean fold RMSE and its test RMSE.

In [ ]:
search = ...
...

print(...)
print(...)
print(...)

<details>
<summary>💡 Hint 1</summary>

`GridSearchCV(DecisionTreeRegressor(random_state=0), {'max_depth': [...]}, cv=folds, scoring='neg_root_mean_squared_error')`, fitted on `train[pair]`.

</details>

<details>
<summary>💡 Hint 2</summary>

The score is negated, so the fold RMSE is `-search.best_score_`.

</details>

<details>
<summary>✅ Solution</summary>

```python
search = GridSearchCV(DecisionTreeRegressor(random_state=0), {'max_depth': [1, 2, 3, 4, 6, 8]},
                      cv=folds, scoring='neg_root_mean_squared_error')
search.fit(train[pair], train['vol_next'])

print(search.best_params_)
print(round(-search.best_score_, 4))
print(round(float(rmse(test['vol_next'], search.predict(test[pair]))), 4))
```

Depth 2, with a mean fold RMSE of 0.6368 and a test RMSE of 0.2691. The fold mean is much larger because the third fold, April 2019 to July 2020, holds the crash of 2020 and scores 1.31 on its own. Keep `search`: D4 continues from here.

</details>

---

### D4 · The fold errors, as a table  ★★★☆☆  · revisits S3

Continuing from D3. Make a pandas Series of the mean fold RMSE for every depth, indexed by the depth and sorted from best to worst. The numbers are in `search.cv_results_`.

In [ ]:
by_depth = ...

print(by_depth)

<details>
<summary>💡 Hint 1</summary>

`search.cv_results_['mean_test_score']` holds the negated scores, and `search.cv_results_['param_max_depth']` the depths in the same order.

</details>

<details>
<summary>💡 Hint 2</summary>

`pd.Series(-scores, index=list(depths)).round(4).sort_values()`.

</details>

<details>
<summary>✅ Solution</summary>

```python
by_depth = pd.Series(-search.cv_results_['mean_test_score'],
                     index=list(search.cv_results_['param_max_depth'])).round(4).sort_values()

print(by_depth)
```

Depth 2 leads with 0.6368, then 4 with 0.6515, and depth 8 is last at 0.7622. The order is not smooth: depth 3 does worse than 4 on these folds, which is why the search tries each value rather than assuming a shape.

</details>

---

### D5 · A minimum leaf size  ★★☆☆☆

Search `min_samples_leaf` over 20, 50, 100, 200 and 400 on the folds, with no depth limit. Print the best value, the depth and number of leaves of the tree it chose, and its test RMSE.

In [ ]:
search_leaf = ...
...

print(...)
print(...)
print(...)

<details>
<summary>💡 Hint</summary>

The chosen tree is `search_leaf.best_estimator_`, with `.get_depth()` and `.get_n_leaves()`.

</details>

<details>
<summary>✅ Solution</summary>

```python
search_leaf = GridSearchCV(DecisionTreeRegressor(random_state=0),
                           {'min_samples_leaf': [20, 50, 100, 200, 400]},
                           cv=folds, scoring='neg_root_mean_squared_error')
search_leaf.fit(train[pair], train['vol_next'])

print(search_leaf.best_params_)
print(search_leaf.best_estimator_.get_depth(), search_leaf.best_estimator_.get_n_leaves())
print(round(float(rmse(test['vol_next'], search_leaf.predict(test[pair]))), 4))
```

Leaves of at least 100 days, a tree of depth 6 with 14 leaves, and a test RMSE of 0.2706. A leaf size lets the tree go deep where days are plentiful and stop where they are not.

</details>

---

### D6 · Pruning with ccp_alpha  ★★★☆☆

Loop over `ccp_alpha` of 0, 0.005, 0.01, 0.02 and 0.05 on `pair`, with no depth limit. Store the number of leaves and the test RMSE of each pruned tree in a dictionary keyed by alpha.

In [ ]:
pruned = {}
for alpha in [0, 0.005, 0.01, 0.02, 0.05]:
    ...

print(pruned)

<details>
<summary>💡 Hint</summary>

`DecisionTreeRegressor(ccp_alpha=alpha, random_state=0)`, then store a tuple of `int(t.get_n_leaves())` and the rounded test RMSE.

</details>

<details>
<summary>✅ Solution</summary>

```python
pruned = {}
for alpha in [0, 0.005, 0.01, 0.02, 0.05]:
    t = DecisionTreeRegressor(ccp_alpha=alpha, random_state=0).fit(train[pair], train['vol_next'])
    pruned[alpha] = (int(t.get_n_leaves()), round(float(rmse(test['vol_next'], t.predict(test[pair]))), 4))

print(pruned)
```

`{0: (1894, 0.4663), 0.005: (12, 0.327), 0.01: (9, 0.303), 0.02: (4, 0.2408), 0.05: (3, 0.2408)}`. A price of 0.02 per leaf cuts 1,894 leaves down to 4, and the test RMSE falls from 0.4663 to 0.2408. At 0.05 two more leaves merge and the score does not move, because no test day lands in either of them. Like ridge's alpha, a larger value gives a simpler model.

</details>

---

---

## E · Averaging, and bagging

A deep tree changes with its sample of days; an average of many does not. These draw the samples and build the average by hand before using scikit-learn.

### E1 · One bootstrap sample  ★★☆☆☆

Draw a bootstrap sample of the training days with `random_state=1`: as many rows as the table, drawn with replacement. Print the share of distinct days in it, and the share the formula 1 minus (1 minus 1/n) to the power n gives.

In [ ]:
n = len(train)
sample = ...
share = ...
formula = ...

print(share)
print(formula)

<details>
<summary>💡 Hint</summary>

`train.sample(n=n, replace=True, random_state=1)`. The distinct days are `sample.index.nunique()`.

</details>

<details>
<summary>✅ Solution</summary>

```python
n = len(train)
sample = train.sample(n=n, replace=True, random_state=1)
share = round(sample.index.nunique() / n, 3)
formula = round(1 - (1 - 1 / n) ** n, 3)

print(share)
print(formula)
```

0.633 against 0.632. About a third of the days are missing from any one sample, and those days are the out-of-bag days a forest can score itself on.

</details>

---

### E2 · Twenty samples  ★★★☆☆  · revisits S2

Repeat E1 for `random_state` 0 to 19. Collect the twenty shares in a list, and print their mean, the smallest and the largest.

In [ ]:
shares = []
for seed in range(20):
    ...

print(...)
print(...)
print(...)

<details>
<summary>💡 Hint</summary>

Inside the loop, draw the sample with `random_state=seed` and append the rounded share. Then `np.mean(shares)`, `min(shares)` and `max(shares)`.

</details>

<details>
<summary>✅ Solution</summary>

```python
shares = []
for seed in range(20):
    s = train.sample(n=len(train), replace=True, random_state=seed)
    shares.append(round(s.index.nunique() / len(train), 3))

print(round(float(np.mean(shares)), 3))
print(min(shares))
print(max(shares))
```

A mean of 0.633, from 0.622 to 0.643. The share hardly moves from sample to sample, which is why 0.632 is quoted as a property of the bootstrap rather than of one draw.

</details>

---

### E3 · How often each day is drawn  ★★★☆☆  · revisits S3

In one bootstrap sample, count how many times each training day was drawn, including the days drawn zero times. Then draw how many days were drawn 0, 1, 2, 3 and more times, as a bar chart.

In [ ]:
sample = train.sample(n=len(train), replace=True, random_state=0)
draws = ...
how_often = ...

fig, ax = plt.subplots(figsize=(7, 3))
...
ax.set_xlabel('times a day was drawn')
ax.set_ylabel('days')
plt.show()

<details>
<summary>💡 Hint 1</summary>

`sample.index.value_counts()` counts the days that were drawn. `.reindex(train.index, fill_value=0)` adds the days that were not.

</details>

<details>
<summary>💡 Hint 2</summary>

`draws.value_counts().sort_index()` then counts how many days were drawn each number of times.

</details>

<details>
<summary>✅ Solution</summary>

```python
sample = train.sample(n=len(train), replace=True, random_state=0)
draws = sample.index.value_counts().reindex(train.index, fill_value=0)
how_often = draws.value_counts().sort_index()

fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(how_often.index, how_often.values, color='#1c5cab')
ax.set_xlabel('times a day was drawn')
ax.set_ylabel('days')
plt.show()
```

687 days were not drawn at all, and one day was drawn 6 times. Each tree of a bagged model sees some days twice or more and a third of them not at all, which is where the differences between the trees come from.

</details>

---

### E4 · Two trees, two halves  ★★★☆☆  · revisits S5

Split the training days into two random halves with `random_state=1`. Grow a full tree on `pair` in each half, then print the mean absolute difference between their test forecasts, the test RMSE of each tree, and the test RMSE of their average.

In [ ]:
half_a = ...
half_b = ...
tree_a = ...
tree_b = ...
...

print(...)
print(...)
print(...)

<details>
<summary>💡 Hint 1</summary>

`half_a = train.sample(frac=0.5, random_state=1)` and `half_b = train.drop(half_a.index)`.

</details>

<details>
<summary>💡 Hint 2</summary>

With `p_a` and `p_b` the two forecasts, the average is `(p_a + p_b) / 2`.

</details>

<details>
<summary>✅ Solution</summary>

```python
half_a = train.sample(frac=0.5, random_state=1)
half_b = train.drop(half_a.index)
tree_a = DecisionTreeRegressor(random_state=0).fit(half_a[pair], half_a['vol_next'])
tree_b = DecisionTreeRegressor(random_state=0).fit(half_b[pair], half_b['vol_next'])
p_a, p_b = tree_a.predict(test[pair]), tree_b.predict(test[pair])

print(round(float(np.abs(p_a - p_b).mean()), 3))
print(round(float(rmse(test['vol_next'], p_a)), 3), round(float(rmse(test['vol_next'], p_b)), 3))
print(round(float(rmse(test['vol_next'], (p_a + p_b) / 2)), 3))
```

The two trees differ by 0.397 on the average test day. Alone they score 0.574 and 0.586; their average scores 0.471. Two noisy forecasts that make different mistakes average out to a better one.

</details>

---

### E5 · Bagging with a list of trees  ★★★☆☆

Grow 25 full trees on the 19 columns, each on its own bootstrap sample with `random_state` 0 to 24, and keep them in a list. Stack their test forecasts into one array and average them with `np.mean(..., axis=0)`. Print the RMSE of the average and of the first tree alone. It takes a few seconds.

In [ ]:
trees = []
for b in range(25):
    ...

forecasts = ...
average = ...

print(...)
print(...)

<details>
<summary>💡 Hint 1</summary>

Inside the loop: draw the sample, fit a `DecisionTreeRegressor(random_state=0)`, and `trees.append(...)` it.

</details>

<details>
<summary>💡 Hint 2</summary>

`forecasts = np.array([t.predict(test[columns]) for t in trees])` has one row per tree, so `axis=0` averages down the columns, day by day.

</details>

<details>
<summary>✅ Solution</summary>

```python
trees = []
for b in range(25):
    s = train.sample(n=len(train), replace=True, random_state=b)
    trees.append(DecisionTreeRegressor(random_state=0).fit(s[columns], s['vol_next']))

forecasts = np.array([t.predict(test[columns]) for t in trees])
average = np.mean(forecasts, axis=0)

print(round(float(rmse(test['vol_next'], average)), 4))
print(round(float(rmse(test['vol_next'], forecasts[0])), 4))
```

0.3169 for the average against 0.4438 for the first tree alone. The array has one row per tree and one column per test day, and `axis=0` is the direction that averages trees rather than days.

</details>

---

### E6 · BaggingRegressor and its first questions  ★★☆☆☆

Fit `BaggingRegressor` with 25 full trees and `random_state=0` on the 19 columns. Print its test RMSE, and count which column each tree asks about first. The fitted trees are in `bag.estimators_`.

In [ ]:
bag = ...
...
first = ...

print(...)
print(...)

<details>
<summary>💡 Hint 1</summary>

`BaggingRegressor(DecisionTreeRegressor(), n_estimators=25, random_state=0)`.

</details>

<details>
<summary>💡 Hint 2</summary>

Loop over `bag.estimators_` and append `columns[t.tree_.feature[0]]` to a list, then `pd.Series(first).value_counts()`.

</details>

<details>
<summary>✅ Solution</summary>

```python
bag = BaggingRegressor(DecisionTreeRegressor(), n_estimators=25, random_state=0)
bag.fit(train[columns], train['vol_next'])
first = []
for t in bag.estimators_:
    first.append(columns[t.tree_.feature[0]])

print(round(float(rmse(test['vol_next'], bag.predict(test[columns]))), 4))
print(pd.Series(first).value_counts())
```

A test RMSE of 0.3055. The first questions are `{'vol_5d': 15, 'DIS_vol': 5, 'XOM_vol': 2, 'vol_10d': 1, 'AAPL_vol': 1, 'ret_5d': 1}`: 24 of the 25 trees open with a volatility column, and 15 with `vol_5d`. Trees that ask alike make mistakes alike, and averaging cancels less of them. That is what a forest's `max_features` sets out to change.

</details>

---

### E7 · Out-of-bag, by hand  ★★★★★

Grow 25 full trees on the 19 columns, each on a bootstrap sample drawn with `random_state` 0 to 24, as in E5. This time, for every training day, add up the forecasts of only the trees whose sample left that day out, and count those trees. Divide to get each day's out-of-bag forecast, and print its RMSE on the training days. It takes a few seconds.

In [ ]:
n = len(train)
total = np.zeros(n)
count = np.zeros(n)
for b in range(25):
    ...

oob = ...

print(...)

<details>
<summary>💡 Hint 1</summary>

`left_out = ~train.index.isin(sample.index)` is True for the days the sample missed. `total[left_out] += tree.predict(train[columns][left_out])` adds the forecasts in the right places.

</details>

<details>
<summary>💡 Hint 2</summary>

`count[left_out] += 1` counts the trees, and `oob = total / count` divides day by day. Every day was left out by at least two trees here, so nothing is divided by zero.

</details>

<details>
<summary>✅ Solution</summary>

```python
n = len(train)
total = np.zeros(n)
count = np.zeros(n)
for b in range(25):
    sample = train.sample(n=n, replace=True, random_state=b)
    tree = DecisionTreeRegressor(random_state=0).fit(sample[columns], sample['vol_next'])
    left_out = ~train.index.isin(sample.index)
    total[left_out] += tree.predict(train[columns][left_out])
    count[left_out] += 1

oob = total / count

print(round(float(rmse(train['vol_next'], oob)), 4))
```

0.2840, with each day left out by 9.2 trees on average. That is better than the 0.3169 the same trees score on the test days, although the training days include 2020 and the test years were calm. A day a tree left out still had its neighbours in the sample, and neighbouring days share most of a 20-day window. F4 puts this number beside the folds.

</details>

---

---

## F · Random forests

A forest is bagging where each question sees only some of the columns. These count what that changes, and check the out-of-bag score against the folds.

### F1 · A forest with a third of the columns  ★☆☆☆☆

Fit a random forest of 100 trees with `max_features=0.33`, a third of the columns at each question, and `random_state=0`. Print its test RMSE and the number of columns each question chooses among.

In [ ]:
forest = ...
...

print(...)
print(...)

<details>
<summary>💡 Hint</summary>

A float `max_features` is a share: each question sees `int(0.33 * len(columns))` columns.

</details>

<details>
<summary>✅ Solution</summary>

```python
forest = RandomForestRegressor(n_estimators=100, max_features=0.33, random_state=0)
forest.fit(train[columns], train['vol_next'])

print(round(float(rmse(test['vol_next'], forest.predict(test[columns]))), 4))
print(int(0.33 * len(columns)))
```

0.2894 with 6 columns per question, against 0.2835 for the lecture's 4 columns. `max_features` is one setting among several, and the next exercises count what it does.

</details>

---

### F2 · How many columns open the trees  ★★★☆☆

For `max_features` of 1, 4 and 19, fit a forest of 100 trees and count how many DIFFERENT columns ask the first question across its trees. Store the counts in a dictionary.

In [ ]:
variety = {}
for m in [1, 4, 19]:
    ...

print(variety)

<details>
<summary>💡 Hint</summary>

Inside the loop, fit the forest, collect `columns[t.tree_.feature[0]]` for every tree in `f.estimators_`, and store `pd.Series(first).nunique()`.

</details>

<details>
<summary>✅ Solution</summary>

```python
variety = {}
for m in [1, 4, 19]:
    f = RandomForestRegressor(n_estimators=100, max_features=m, random_state=0)
    f.fit(train[columns], train['vol_next'])
    first = [columns[t.tree_.feature[0]] for t in f.estimators_]
    variety[m] = int(pd.Series(first).nunique())

print(variety)
```

`{1: 19, 4: 12, 19: 7}`. With all 19 columns to choose from, the 100 trees open with only 7 different columns. With one column drawn at random for each question, 19 of the 19 open some tree. Less choice at each question is what makes the trees differ.

</details>

---

### F3 · The error as trees are added  ★★★★☆  · revisits S3

Fit a forest of 200 trees with `max_features='sqrt'`. Take each tree's test forecasts from `forest.estimators_`, and compute the RMSE of the average of the first 1, 2, ..., 200 trees. Draw it on a log x-axis, with a dashed line at linear regression's test RMSE on `vol_20d`.

In [ ]:
forest = RandomForestRegressor(n_estimators=200, max_features='sqrt', random_state=0)
forest.fit(train[columns], train['vol_next'])

each = ...
running = ...
errors = ...
line = ...

fig, ax = plt.subplots(figsize=(8, 3))
...
...
ax.set_xscale('log')
ax.set_xlabel('trees averaged')
ax.set_ylabel('test RMSE')
plt.show()

<details>
<summary>💡 Hint 1</summary>

`each = np.array([e.predict(test[columns].to_numpy()) for e in forest.estimators_])` has one row per tree. `each.cumsum(axis=0)` adds the trees up in order; divide row n by n + 1 for the running average.

</details>

<details>
<summary>💡 Hint 2</summary>

`running = each.cumsum(axis=0) / np.arange(1, 201)[:, None]`, then one RMSE per row. The dashed line is `ax.axhline(line, linestyle='--')`.

</details>

<details>
<summary>✅ Solution</summary>

```python
forest = RandomForestRegressor(n_estimators=200, max_features='sqrt', random_state=0)
forest.fit(train[columns], train['vol_next'])

each = np.array([e.predict(test[columns].to_numpy()) for e in forest.estimators_])
running = each.cumsum(axis=0) / np.arange(1, 201)[:, None]
errors = [rmse(test['vol_next'], row) for row in running]
line = rmse(test['vol_next'], LinearRegression().fit(train[['vol_20d']], train['vol_next'])
            .predict(test[['vol_20d']]))

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(range(1, 201), errors, color='#1c5cab', label='random forest')
ax.axhline(line, color='grey', linestyle='--', label='linear regression on vol_20d')
ax.set_xscale('log')
ax.set_xlabel('trees averaged')
ax.set_ylabel('test RMSE')
ax.legend()
plt.show()
```

The error falls fast over the first ten trees, from 0.4271 to 0.2978, then drifts down with small wiggles: 0.2835 at 100 and 0.2764 at 200. It stays above the dashed line at 0.2369 throughout. Adding trees does not overfit; it only stops helping, and every tree still costs time.

</details>

---

### F4 · Out-of-bag on days  ★★★☆☆

Fit a forest of 100 trees with `max_features='sqrt'`, `min_samples_leaf=50` and `oob_score=True`. Print its out-of-bag RMSE on the training days, its mean RMSE on the time-series folds, and its test RMSE. The folds take a few seconds.

In [ ]:
forest = ...
...
oob = ...
on_folds = ...

print(oob)
print(on_folds)
print(...)

<details>
<summary>💡 Hint 1</summary>

`forest.oob_prediction_` holds a forecast for every training day from the trees that did not see it: `rmse(train['vol_next'], forest.oob_prediction_)`.

</details>

<details>
<summary>💡 Hint 2</summary>

`cross_val_score(forest, train[columns], train['vol_next'], cv=folds, scoring='neg_root_mean_squared_error')`, negated and averaged.

</details>

<details>
<summary>✅ Solution</summary>

```python
forest = RandomForestRegressor(n_estimators=100, max_features='sqrt', min_samples_leaf=50,
                               oob_score=True, random_state=0)
forest.fit(train[columns], train['vol_next'])
oob = round(float(rmse(train['vol_next'], forest.oob_prediction_)), 4)
on_folds = round(float(-cross_val_score(forest, train[columns], train['vol_next'], cv=folds,
                                        scoring='neg_root_mean_squared_error').mean()), 4)

print(oob)
print(on_folds)
print(round(float(rmse(test['vol_next'], forest.predict(test[columns]))), 4))
```

0.5034 out-of-bag against 0.6158 on folds that score only later days, and 0.2482 on the test days. Larger leaves narrow the gap the lecture showed for full trees, but out-of-bag still flatters: a day left out of a tree has its neighbours in the bag.

</details>

---

### F5 · max_features on the folds  ★★★☆☆  · revisits S5

Score forests of 100 trees with `min_samples_leaf=50` and `max_features` of 1, 2, 4, 8 and 19 on the time-series folds. Store each one's mean fold RMSE in a dictionary and print the best setting. It takes a few seconds.

In [ ]:
by_features = {}
for m in [1, 2, 4, 8, 19]:
    ...

best = ...

print(by_features)
print(best)

<details>
<summary>💡 Hint 1</summary>

Inside the loop, build the forest and score it with `cross_val_score(f, train[columns], train['vol_next'], cv=folds, scoring='neg_root_mean_squared_error')`, negated and averaged.

</details>

<details>
<summary>💡 Hint 2</summary>

`min(by_features, key=by_features.get)` gives the key with the smallest value.

</details>

<details>
<summary>✅ Solution</summary>

```python
by_features = {}
for m in [1, 2, 4, 8, 19]:
    f = RandomForestRegressor(n_estimators=100, max_features=m, min_samples_leaf=50, random_state=0)
    scores = cross_val_score(f, train[columns], train['vol_next'], cv=folds,
                             scoring='neg_root_mean_squared_error')
    by_features[m] = round(float(-scores.mean()), 4)

best = min(by_features, key=by_features.get)

print(by_features)
print(best)
```

`{1: 0.5918, 2: 0.5976, 4: 0.6158, 8: 0.6519, 19: 0.703}`, best at 1. The error rises with every column added to the choice, and at 19 the forest is bagging with leaves of 50, the worst of the five. The folds choose the setting, so the test days stay unopened for the model you keep. F6 adds a second setting to the same loop.

</details>

---

### F6 · Two settings in one loop  ★★★★☆  · revisits S2

Score forests of 100 trees on the time-series folds for every pair of `max_features` in [1, 4] and `min_samples_leaf` in [20, 50, 100], with two nested loops. Store the mean fold RMSE in a dictionary keyed by the pair, and print the best pair. It takes about ten seconds.

In [ ]:
scores = {}
for m in [1, 4]:
    for leaf in [20, 50, 100]:
        ...

best_pair = ...

print(scores)
print(best_pair)

<details>
<summary>💡 Hint</summary>

Key the dictionary with a tuple, `scores[(m, leaf)] = ...`, and score with `cross_val_score(..., cv=folds, scoring='neg_root_mean_squared_error')`.

</details>

<details>
<summary>✅ Solution</summary>

```python
scores = {}
for m in [1, 4]:
    for leaf in [20, 50, 100]:
        f = RandomForestRegressor(n_estimators=100, max_features=m, min_samples_leaf=leaf,
                                  random_state=0)
        scores[(m, leaf)] = round(float(-cross_val_score(f, train[columns], train['vol_next'],
                                  cv=folds, scoring='neg_root_mean_squared_error').mean()), 4)

best_pair = min(scores, key=scores.get)

print(scores)
print(best_pair)
```

The best pair is `(1, 50)` with 0.5918. Two nested loops over a dictionary of results is exactly what `GridSearchCV` does for you, which section G uses next.

</details>

---

---

## G · Tuning, and what it costs

A grid multiplies: every setting you add multiplies the fits. These count, time and search grids, and check whether a bigger forest is a better one.

### G1 · How big is a grid  ★☆☆☆☆  · revisits S2

A grid is a dictionary of lists. With a loop, compute how many combinations the grid below has, and how many fits that means on five folds.

In [ ]:
grid = {'max_features': [1, 2, 4, 8, 19],
        'min_samples_leaf': [5, 20, 50, 100, 200, 400],
        'n_estimators': [100, 300, 500],
        'max_depth': [3, 5, 10, None]}

combinations = 1
for values in grid.values():
    ...

fits = ...

print(combinations)
print(fits)

<details>
<summary>💡 Hint</summary>

Inside the loop: `combinations = combinations * len(values)`. The fits are the combinations times 5.

</details>

<details>
<summary>✅ Solution</summary>

```python
grid = {'max_features': [1, 2, 4, 8, 19],
        'min_samples_leaf': [5, 20, 50, 100, 200, 400],
        'n_estimators': [100, 300, 500],
        'max_depth': [3, 5, 10, None]}

combinations = 1
for values in grid.values():
    combinations = combinations * len(values)

fits = combinations * 5

print(combinations)
print(fits)
```

360 combinations and 1,800 fits, each a forest of up to 500 trees. This is the grid that took 14 minutes on one core in the lecture; adding one more list of four values would make it an hour.

</details>

---

### G2 · Timing one fit  ★★☆☆☆

Time one fit of a forest of 100 trees with `max_features='sqrt'`, using `time.perf_counter()` before and after. Then estimate, in minutes, how long the 1,800 fits of G1 would take if every fit took as long.

In [ ]:
forest = RandomForestRegressor(n_estimators=100, max_features='sqrt', random_state=0)

start = ...
...
seconds = ...
minutes = ...

print(seconds)
print(minutes)

<details>
<summary>💡 Hint</summary>

`start = time.perf_counter()`, then the fit, then `seconds = time.perf_counter() - start`.

</details>

<details>
<summary>✅ Solution</summary>

```python
forest = RandomForestRegressor(n_estimators=100, max_features='sqrt', random_state=0)

start = time.perf_counter()
forest.fit(train[columns], train['vol_next'])
seconds = round(time.perf_counter() - start, 2)
minutes = round(seconds * 1800 / 60, 1)

print(seconds)
print(minutes)
```

The lecture's machine took 0.57 seconds, which makes about 17 minutes for the grid; your number depends on your machine. The estimate is rough, because 300 or 500 trees take longer and shallow trees take less, but it tells you before you start whether to wait or to shrink the grid.

</details>

---

### G3 · A grid as three lines  ★★★★☆

Search `max_features` in [1, 4, 19] and `min_samples_leaf` in [5, 20, 50, 100, 200] for a forest of 50 trees on the folds. Put the mean fold RMSE into a table with one column per `max_features`, and draw one line per column against the leaf size on a log axis. It takes up to half a minute.

In [ ]:
search = ...
...
results = ...

fig, ax = plt.subplots(figsize=(8, 3))
...
ax.set_xscale('log')
ax.set_xlabel('min_samples_leaf')
ax.set_ylabel('mean fold RMSE')
plt.show()

<details>
<summary>💡 Hint 1</summary>

`pd.DataFrame(search.cv_results_).pivot(index='param_min_samples_leaf', columns='param_max_features', values='mean_test_score')` gives the table; multiply by -1.

</details>

<details>
<summary>💡 Hint 2</summary>

`for m in results.columns: ax.plot(results.index, results[m], marker='o', label=f'max_features={m}')`, then `ax.legend()`.

</details>

<details>
<summary>✅ Solution</summary>

```python
search = GridSearchCV(RandomForestRegressor(n_estimators=50, random_state=0),
                      {'max_features': [1, 4, 19], 'min_samples_leaf': [5, 20, 50, 100, 200]},
                      cv=folds, scoring='neg_root_mean_squared_error')
search.fit(train[columns], train['vol_next'])
results = pd.DataFrame(search.cv_results_).pivot(index='param_min_samples_leaf',
                                                 columns='param_max_features',
                                                 values='mean_test_score') * -1

fig, ax = plt.subplots(figsize=(8, 3))
for m in results.columns:
    ax.plot(results.index, results[m], marker='o', label=f'max_features={m}')
ax.set_xscale('log')
ax.set_xlabel('min_samples_leaf')
ax.set_ylabel('mean fold RMSE')
ax.legend()
plt.show()
```

The best pair is `{'max_features': 1, 'min_samples_leaf': 50}` with 0.5926, and the worst cell is 0.8041. The line for all 19 columns sits highest at every leaf size, and each line is lowest at a leaf size of 50 or 100, not at either end. Both settings matter, and they matter together.

</details>

---

### G4 · A random sample of a grid  ★★★☆☆

`RandomizedSearchCV` tries `n_iter` combinations drawn at random instead of all of them. Search the grid below with `n_iter=10`, `random_state=0` and a forest of 50 trees on the folds. Print the best settings, the mean fold RMSE and the test RMSE. It takes about ten seconds.

In [ ]:
grid = {'max_features': [1, 2, 4, 8, 19],
        'min_samples_leaf': [5, 20, 50, 100, 200, 400],
        'max_depth': [3, 5, 10, None]}

sampled = ...
...

print(...)
print(...)
print(...)

<details>
<summary>💡 Hint</summary>

`RandomizedSearchCV(RandomForestRegressor(n_estimators=50, random_state=0), grid, n_iter=10, cv=folds, scoring='neg_root_mean_squared_error', random_state=0)`. It is read like a `GridSearchCV`.

</details>

<details>
<summary>✅ Solution</summary>

```python
grid = {'max_features': [1, 2, 4, 8, 19],
        'min_samples_leaf': [5, 20, 50, 100, 200, 400],
        'max_depth': [3, 5, 10, None]}

sampled = RandomizedSearchCV(RandomForestRegressor(n_estimators=50, random_state=0), grid,
                             n_iter=10, cv=folds, scoring='neg_root_mean_squared_error',
                             random_state=0)
sampled.fit(train[columns], train['vol_next'])

print(sampled.best_params_)
print(round(-sampled.best_score_, 4))
print(round(float(rmse(test['vol_next'], sampled.predict(test[columns]))), 4))
```

`{'min_samples_leaf': 50, 'max_features': 2, 'max_depth': 3}`, with 0.5885 on the folds and 0.2266 on the test days. These are the settings the lecture's 14-minute search chose, found here from 10 of the 120 combinations. A random sample will not always land on the best corner, but it usually lands near it, at a fraction of the cost.

</details>

---

### G5 · More trees  ★★☆☆☆  · revisits S2

Fit forests with `max_features='sqrt'`, `min_samples_leaf=50` and 25, 50, 100, 200 and 400 trees. Print one line per forest with the number of trees and the test RMSE, lined up with field widths.

In [ ]:
for n in [25, 50, 100, 200, 400]:
    ...

<details>
<summary>💡 Hint</summary>

Fit inside the loop, then `print(f"{n:>5}{error:>9.4f}")`.

</details>

<details>
<summary>✅ Solution</summary>

```python
for n in [25, 50, 100, 200, 400]:
    f = RandomForestRegressor(n_estimators=n, max_features='sqrt', min_samples_leaf=50, random_state=0)
    f.fit(train[columns], train['vol_next'])
    error = rmse(test['vol_next'], f.predict(test[columns]))
    print(f"{n:>5}{error:>9.4f}")
```

From 0.2492 with 25 trees to 0.2463 with 400, and 0.2482 at 100. Sixteen times the trees buy almost nothing, and the fit takes sixteen times as long.

</details>

---

### G6 · Every core  ★★☆☆☆

Time the same forest of 200 trees twice: once with `n_jobs=1` and once with `n_jobs=-1`, which uses every core. Print both times and how many times faster the second was.

In [ ]:
times = {}
for jobs in [1, -1]:
    ...

print(times)
print(...)

<details>
<summary>💡 Hint</summary>

Inside the loop, build the forest with `n_jobs=jobs`, time the fit with `time.perf_counter()`, and store the seconds in `times[jobs]`.

</details>

<details>
<summary>✅ Solution</summary>

```python
times = {}
for jobs in [1, -1]:
    f = RandomForestRegressor(n_estimators=200, max_features='sqrt', random_state=0, n_jobs=jobs)
    start = time.perf_counter()
    f.fit(train[columns], train['vol_next'])
    times[jobs] = round(time.perf_counter() - start, 2)

print(times)
print(round(times[1] / times[-1], 1))
```

The speed-up depends on how many cores your machine has, and it is always less than that number, because handing out the work takes time too. The trees of a forest are independent, so they can be grown side by side, and the forecasts are identical either way.

</details>

---

---

## H · Trees for a label

The same cuts, with the share of each class in every leaf. These use the credit table, split as in Session 9, and start from the error of a box of 0s and 1s.

### H1 · The error of a box of labels  ★★★☆☆

For labels of 0 and 1, a box with n rows and k ones has an RSS of k(n minus k)/n, and a Gini of 2p(1 minus p), where p = k/n is the share of ones. Check both on the credit training rows: compute the RSS of `default` directly and from the formula, then Gini, and show that Gini is twice the RSS per row.

In [ ]:
y = c_train['default']

n = ...
k = ...
direct = ...
formula = ...
gini = ...
twice_per_row = ...

print(direct, formula)
print(gini, twice_per_row)

<details>
<summary>💡 Hint 1</summary>

`n = len(y)` and `k = int(y.sum())`. The direct RSS is `((y - y.mean()) ** 2).sum()`, the same line as A1 with `y` in place of the array.

</details>

<details>
<summary>💡 Hint 2</summary>

With `p = k / n`, Gini is `2 * p * (1 - p)`, and twice the RSS per row is `2 * direct / n`. Round each so the pairs can be compared.

</details>

<details>
<summary>✅ Solution</summary>

```python
y = c_train['default']

n = len(y)
k = int(y.sum())
direct = round(float(((y - y.mean()) ** 2).sum()), 2)
formula = round(k * (n - k) / n, 2)
gini = round(2 * (k / n) * (1 - k / n), 4)
twice_per_row = round(2 * direct / n, 4)

print(direct, formula)
print(gini, twice_per_row)
```

Both RSS values are 3,617.57, from 4,645 defaults among 21,000 borrowers, and both of the others are 0.3445: the 0.345 that `plot_tree` prints in the top box of a credit tree. A box of labels needs only its two counts to know its error. Gini is the squared error per row, doubled, so a classification tree chooses the same cuts a regression tree would on the 0s and 1s.

</details>

---

### H2 · A classification tree  ★☆☆☆☆  · revisits S9

Fit a `DecisionTreeClassifier` of depth 2 on the seven credit columns. Print its test AUC and the distinct probabilities of default it gives.

In [ ]:
c_tree = ...
...
p_tree = ...

print(...)
print(...)

<details>
<summary>💡 Hint</summary>

`p_tree = c_tree.predict_proba(c_test[c_columns])[:, 1]`, then `roc_auc_score(c_test['default'], p_tree)` and `np.unique(p_tree).round(3)`.

</details>

<details>
<summary>✅ Solution</summary>

```python
c_tree = DecisionTreeClassifier(max_depth=2, random_state=0)
c_tree.fit(c_train[c_columns], c_train['default'])
p_tree = c_tree.predict_proba(c_test[c_columns])[:, 1]

print(round(roc_auc_score(c_test['default'], p_tree), 4))
print(np.unique(p_tree).round(3))
```

An AUC of 0.7224 from four probabilities, [0.116, 0.309, 0.41, 0.705]. Four distinct numbers can rank 9,000 borrowers only in four groups, which caps how high a shallow tree's AUC can go.

</details>

---

### H3 · Draw the classification tree  ★☆☆☆☆

Fit a classification tree of depth 2 on the seven credit columns, and draw it with shares instead of counts, filled boxes, no Gini line, and the classes named `'repaid'` and `'defaulted'` with `class_names`.

In [ ]:
c_tree = ...
...

plt.figure(figsize=(11, 3))
...
plt.show()

<details>
<summary>💡 Hint</summary>

`plot_tree(c_tree, feature_names=c_columns, filled=True, proportion=True, impurity=False, class_names=['repaid', 'defaulted'], fontsize=9)`.

</details>

<details>
<summary>✅ Solution</summary>

```python
c_tree = DecisionTreeClassifier(max_depth=2, random_state=0)
c_tree.fit(c_train[c_columns], c_train['default'])

plt.figure(figsize=(11, 3))
plot_tree(c_tree, feature_names=c_columns, filled=True, proportion=True, impurity=False,
          class_names=['repaid', 'defaulted'], fontsize=9)
plt.show()
```

The first question is `late_now <= 1.5`. Borrowers up to one month behind are then asked about `months_late`, and those two or more months behind about `utilisation`: the 83 of them who used almost none of their limit defaulted at 0.41, the rest at 0.71. Only that last leaf has `defaulted` as its larger class.

</details>

---

### H4 · How much the first question lowers Gini  ★★★☆☆

Compute Gini, 2p(1 minus p), for all credit training rows, and for the two boxes of the first question, `late_now` of 1 or less and above 1. Then compute the average of the two boxes' Gini weighted by their sizes.

In [ ]:
def gini(labels):
    """..."""
    ...


left = c_train[c_train['late_now'] <= 1]['default']
right = c_train[c_train['late_now'] > 1]['default']

root = ...
weighted = ...

print(root)
print(weighted)

<details>
<summary>💡 Hint 1</summary>

`gini` returns `2 * p * (1 - p)` with `p = labels.mean()`.

</details>

<details>
<summary>💡 Hint 2</summary>

The weighted average is `(len(left) * gini(left) + len(right) * gini(right)) / len(c_train)`.

</details>

<details>
<summary>✅ Solution</summary>

```python
def gini(labels):
    """Gini of a box of 0/1 labels: 2p(1 - p)."""
    p = labels.mean()
    return 2 * p * (1 - p)


left = c_train[c_train['late_now'] <= 1]['default']
right = c_train[c_train['late_now'] > 1]['default']

root = round(float(gini(c_train['default'])), 4)
weighted = round(float((len(left) * gini(left) + len(right) * gini(right)) / len(c_train)), 4)

print(root)
print(weighted)
```

From 0.3445 to 0.2919: the left box has a Gini of 0.2764 and the right box 0.4245. The right box is less pure than the whole table, but it holds only 2,208 borrowers, 10.5 percent, while the left box, with everyone else, became purer. That fall in the weighted average is what the tree made as large as it could when it chose this question.

</details>

---

### H5 · A forest for the label  ★★★☆☆  · revisits S8

Fit a random forest of 100 trees with `min_samples_leaf=50` and `oob_score=True` on the credit table. Print its out-of-bag AUC, its test AUC, and the test AUC of a scaled logistic regression beside them.

In [ ]:
c_forest = ...
...
logistic = ...
...

print(...)
print(...)
print(...)

<details>
<summary>💡 Hint 1</summary>

The out-of-bag probabilities are `c_forest.oob_decision_function_[:, 1]`, scored against `c_train['default']`.

</details>

<details>
<summary>💡 Hint 2</summary>

`Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])` for the comparison.

</details>

<details>
<summary>✅ Solution</summary>

```python
c_forest = RandomForestClassifier(n_estimators=100, min_samples_leaf=50, oob_score=True,
                                  random_state=0)
c_forest.fit(c_train[c_columns], c_train['default'])
logistic = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
logistic.fit(c_train[c_columns], c_train['default'])

print(round(roc_auc_score(c_train['default'], c_forest.oob_decision_function_[:, 1]), 4))
print(round(roc_auc_score(c_test['default'], c_forest.predict_proba(c_test[c_columns])[:, 1]), 4))
print(round(roc_auc_score(c_test['default'], logistic.predict_proba(c_test[c_columns])[:, 1]), 4))
```

0.780 out-of-bag and 0.776 on the test borrowers, against 0.748 for logistic regression. Here the out-of-bag score is close to the test score, because the rows are separate borrowers and a left-out borrower has no neighbour in the bag.

</details>

---

### H6 · The default rate in each leaf  ★★★☆☆  · revisits S3

Fit the depth-2 classification tree, find the leaf of every training borrower with `.apply()`, and draw the share that defaulted in each leaf as bars, with a dashed line at the share for everyone.

In [ ]:
c_tree = ...
...

leaf = ...
by_leaf = ...

fig, ax = plt.subplots(figsize=(7, 3))
...
...
ax.set_xlabel('leaf')
ax.set_ylabel('share that defaulted')
plt.show()

<details>
<summary>💡 Hint 1</summary>

`leaf = pd.Series(c_tree.apply(c_train[c_columns]), index=c_train.index)`, then `c_train['default'].groupby(leaf).mean()`.

</details>

<details>
<summary>💡 Hint 2</summary>

Draw the bars against the leaf numbers as text, `by_leaf.index.astype(str)`, so they sit side by side.

</details>

<details>
<summary>✅ Solution</summary>

```python
c_tree = DecisionTreeClassifier(max_depth=2, random_state=0)
c_tree.fit(c_train[c_columns], c_train['default'])

leaf = pd.Series(c_tree.apply(c_train[c_columns]), index=c_train.index)
by_leaf = c_train['default'].groupby(leaf).mean()

fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(by_leaf.index.astype(str), by_leaf.values, color='#1c5cab')
ax.axhline(c_train['default'].mean(), color='grey', linestyle='--')
ax.set_xlabel('leaf')
ax.set_ylabel('share that defaulted')
plt.show()
```

The four leaves default at 0.12, 0.31, 0.41 and 0.71, around the dashed line at 0.221. Each bar is exactly the probability the tree gives every borrower in that leaf.

</details>

---

---

## I · When it goes wrong

Five mistakes that are easy to make with trees. Two of them stop with an error. Three run quietly and print a wrong number, which is worse. In each, the first cell shows the mistake and the second is for your fix.

### I1 · One bracket too few  ★☆☆☆☆  · revisits S3

The cell below fits a stump on one column and raises an error. Run it and read the last line of the message. Then fit the stump correctly in the second cell and print its cut.

In [ ]:
stump = DecisionTreeRegressor(max_depth=1, random_state=0)
stump.fit(train['vol_20d'], train['vol_next'])

In [ ]:
stump = ...
...

print(...)

<details>
<summary>💡 Hint 1</summary>

`train['vol_20d']` is a Series: one column of numbers. scikit-learn wants a table of columns, even when there is only one.

</details>

<details>
<summary>💡 Hint 2</summary>

`train[['vol_20d']]`, with a list inside the brackets, is a DataFrame with one column.

</details>

<details>
<summary>✅ Solution</summary>

```python
stump = DecisionTreeRegressor(max_depth=1, random_state=0)
stump.fit(train[['vol_20d']], train['vol_next'])

print(round(float(stump.tree_.threshold[0]), 4))
```

A `ValueError` asking for a 2-dimensional input, then 0.9859. Single brackets give a Series and double brackets a DataFrame. The target may be a Series; the columns the model asks about may not.

</details>

---

### I2 · The columns in another order  ★★☆☆☆

The tree below was fitted on `vol_20d` and `ret_20d`, and its last line forecasts the test days with the same two columns in the other order. Run it and read the message. Then refit the tree in the second cell, forecast the test days correctly, and print the test RMSE.

In [ ]:
small = DecisionTreeRegressor(max_depth=2, random_state=0)
small.fit(train[['vol_20d', 'ret_20d']], train['vol_next'])
small.predict(test[['ret_20d', 'vol_20d']])

In [ ]:
small = ...
...
forecasts = ...

print(...)

<details>
<summary>💡 Hint</summary>

Give `predict` the columns in the order the tree was fitted on: `test[pair]`, or the same list written out.

</details>

<details>
<summary>✅ Solution</summary>

```python
small = DecisionTreeRegressor(max_depth=2, random_state=0)
small.fit(train[pair], train['vol_next'])
forecasts = small.predict(test[pair])

print(round(float(rmse(test['vol_next'], forecasts)), 4))
```

A `ValueError`: the feature names must be in the same order as in the fit. Then 0.2691. A tree stores its questions by column position, so swapped columns would ask the `vol_20d` question of `ret_20d`, and scikit-learn checks the names to stop that. Keeping the column list in one variable, like `pair`, avoids the mistake.

</details>

---

### I3 · A perfect score  ★★☆☆☆  · revisits S5

The cell below reports a test RMSE of zero. Nothing raised, and the number is wrong. Find the mistake, and fix it in the second cell.

In [ ]:
full = DecisionTreeRegressor(random_state=0)
full.fit(table[pair], table['vol_next'])
print(round(float(rmse(test['vol_next'], full.predict(test[pair]))), 4))

In [ ]:
full = ...
...

print(...)

<details>
<summary>💡 Hint</summary>

Which rows was the tree fitted on, and which rows is it scored on?

</details>

<details>
<summary>✅ Solution</summary>

```python
full = DecisionTreeRegressor(random_state=0)
full.fit(train[pair], train['vol_next'])

print(round(float(rmse(test['vol_next'], full.predict(test[pair]))), 4))
```

0.0 becomes 0.4663. The tree was fitted on `table`, which holds the test days too, and a tree with no limit reproduces every day it was fitted on. A score that looks too good is a reason to check the split before anything else.

</details>

---

### I4 · A random forest that is not random  ★★☆☆☆  · revisits S6

The cell below fits what its author thought was a random forest, and it scores worse than the lecture's. Nothing raised. Read the forest's settings with `get_params()` to find out why, then fit a real random forest in the second cell and print the test RMSE of both.

In [ ]:
forest = RandomForestRegressor(n_estimators=100, random_state=0)
forest.fit(train[columns], train['vol_next'])
print(rmse(test['vol_next'], forest.predict(test[columns])))

In [ ]:
print(...)

real = ...
...
print(...)

<details>
<summary>💡 Hint 1</summary>

`forest.get_params()['max_features']` is how many columns each question may choose among. A float is a share of the columns.

</details>

<details>
<summary>💡 Hint 2</summary>

`RandomForestRegressor(n_estimators=100, max_features='sqrt', random_state=0)` is the forest the lecture used.

</details>

<details>
<summary>✅ Solution</summary>

```python
forest = RandomForestRegressor(n_estimators=100, random_state=0)
forest.fit(train[columns], train['vol_next'])
print(forest.get_params()['max_features'])

real = RandomForestRegressor(n_estimators=100, max_features='sqrt', random_state=0)
real.fit(train[columns], train['vol_next'])
print(rmse(test['vol_next'], forest.predict(test[columns])), rmse(test['vol_next'], real.predict(test[columns])))
```

`max_features` is 1.0 by default for a forest on a number: every question sees every column, which is bagging, not a random forest. It scores 0.3027 against 0.2835, and its 100 trees open with 7 different columns against 12. A forest for a label defaults to `'sqrt'`, which is what makes the mistake easy: the same name, a different default. Read the settings before trusting a name.

</details>

---

### I5 · A forest that changes every run  ★★☆☆☆

The cell below fits the same forest twice and prints two different test RMSEs. Run it again and watch them move. In the second cell, fit both forests so that they agree, and write an `assert` that checks it.

In [ ]:
first = RandomForestRegressor(n_estimators=50, max_features='sqrt')
second = RandomForestRegressor(n_estimators=50, max_features='sqrt')
first.fit(train[columns], train['vol_next'])
second.fit(train[columns], train['vol_next'])
print(rmse(test['vol_next'], first.predict(test[columns])))
print(rmse(test['vol_next'], second.predict(test[columns])))

In [ ]:
first = ...
second = ...
...
score_first = ...
score_second = ...
...

print(score_first)
print(score_second)

<details>
<summary>💡 Hint 1</summary>

Each forest draws its bootstrap samples and its columns at random. `random_state` fixes the draws.

</details>

<details>
<summary>💡 Hint 2</summary>

`assert score_first == score_second` stops the cell with an `AssertionError` if the two differ.

</details>

<details>
<summary>✅ Solution</summary>

```python
first = RandomForestRegressor(n_estimators=50, max_features='sqrt', random_state=0)
second = RandomForestRegressor(n_estimators=50, max_features='sqrt', random_state=0)
first.fit(train[columns], train['vol_next'])
second.fit(train[columns], train['vol_next'])
score_first = rmse(test['vol_next'], first.predict(test[columns]))
score_second = rmse(test['vol_next'], second.predict(test[columns]))
assert score_first == score_second

print(score_first)
print(score_second)
```

Without a seed the two numbers differ, often in the second decimal, and differently on every run. With `random_state=0` both forests draw the same samples and the same columns, so they are the same forest. Fix the seed whenever a number goes into a report, so that it can be reproduced.

</details>

---

---

## J · The desk, one column at a time

The eleven instruments, forecast with one or two columns by a tree and by a line. J2 to J5 use the function from J1. Returns are in percent, from `rets` in the setup cell.

### J1 · A table per instrument  ★★★☆☆  · revisits S2

Write `vol_table(ticker)` returning a DataFrame with the columns `vol_20d`, `ret_20d` and `vol_next` for one instrument, with incomplete rows dropped. Test it on `'AAPL'`.

In [ ]:
def vol_table(ticker):
    """..."""
    ...


print(vol_table('AAPL'))

<details>
<summary>💡 Hint 1</summary>

`r = rets[ticker]`, then `r.rolling(20).std()` and `r.rolling(20).mean()`.

</details>

<details>
<summary>💡 Hint 2</summary>

The target looks forward: `r.rolling(20).std().shift(-20)`. End with `.dropna()`.

</details>

<details>
<summary>✅ Solution</summary>

```python
def vol_table(ticker):
    """vol_20d, ret_20d and vol_next for one instrument, in percent."""
    r = rets[ticker]
    frame = pd.DataFrame({'vol_20d': r.rolling(20).std(), 'ret_20d': r.rolling(20).mean()})
    frame['vol_next'] = r.rolling(20).std().shift(-20)
    return frame.dropna()


print(vol_table('AAPL'))
```

2,476 rows and 3 columns for Apple. One function, eleven instruments: writing the table once is what keeps the next four exercises short.

</details>

---

### J2 · A stump against a line  ★★★☆☆  · revisits S5

Using `vol_table` from J1, fit a stump and a linear regression on `vol_20d` for every instrument, split at the end of 2022. Store both test RMSEs in dictionaries, and print the instruments where the stump wins.

In [ ]:
stump_err = {}
line_err = {}
for ticker in tickers:
    ...

wins = ...

print(wins)

<details>
<summary>💡 Hint</summary>

Inside the loop: `frame = vol_table(ticker)`, split with `.loc`, fit both models on `early[['vol_20d']]`, and store the rounded test RMSEs.

</details>

<details>
<summary>✅ Solution</summary>

```python
stump_err = {}
line_err = {}
for ticker in tickers:
    frame = vol_table(ticker)
    early, late = frame.loc[:'2022-12-31'], frame.loc['2023-01-01':]
    s = DecisionTreeRegressor(max_depth=1, random_state=0).fit(early[['vol_20d']], early['vol_next'])
    l = LinearRegression().fit(early[['vol_20d']], early['vol_next'])
    stump_err[ticker] = round(float(rmse(late['vol_next'], s.predict(late[['vol_20d']]))), 4)
    line_err[ticker] = round(float(rmse(late['vol_next'], l.predict(late[['vol_20d']]))), 4)

wins = [t for t in tickers if stump_err[t] < line_err[t]]

print(wins)
```

The stump wins on 2 of the 11, JNJ and PG. One question gives two forecasts, and on most instruments two numbers are not enough to beat a slope.

</details>

---

### J3 · Where each stump cuts  ★★★☆☆

Using `vol_table` from J1, fit a stump on `vol_20d` for every instrument's training days. Store its cut and the number of training days above the cut as a tuple in a dictionary, and print one line per instrument, sorted from the lowest cut to the highest.

In [ ]:
stumps = {}
for ticker in tickers:
    ...

for ticker in sorted(stumps, key=stumps.get):
    ...

<details>
<summary>💡 Hint 1</summary>

Inside the loop: `cut = s.tree_.threshold[0]`, then `stumps[ticker] = (round(float(cut), 3), int((early['vol_20d'] > cut).sum()))`.

</details>

<details>
<summary>💡 Hint 2</summary>

Tuples sort by their first element, so `sorted(stumps, key=stumps.get)` orders the instruments by the cut.

</details>

<details>
<summary>✅ Solution</summary>

```python
stumps = {}
for ticker in tickers:
    early = vol_table(ticker).loc[:'2022-12-31']
    s = DecisionTreeRegressor(max_depth=1, random_state=0).fit(early[['vol_20d']], early['vol_next'])
    cut = s.tree_.threshold[0]
    stumps[ticker] = (round(float(cut), 3), int((early['vol_20d'] > cut).sum()))

for ticker in sorted(stumps, key=stumps.get):
    print(ticker, stumps[ticker])
```

From WMT at 0.763 to JPM at 3.777. For JPM, KO and PG the busy box holds only 38 to 46 days, all in the spring of 2020: their one question is spent on the crash. WMT's stump does the opposite and sets aside its 438 calmest days. The same question means different things on different instruments.

</details>

---

### J4 · Two questions against a line  ★★★★★

Using `vol_table` from J1, fit a tree of depth 2 and a linear regression on `vol_20d` and `ret_20d` for every instrument. Print the instruments where the tree wins, sorted by the size of the win as a percentage of the line's RMSE, each with that percentage.

In [ ]:
tree_err = {}
line_err = {}
for ticker in tickers:
    ...

wins = ...
...

<details>
<summary>💡 Hint 1</summary>

The same loop as J2 with `pair` as the columns and `max_depth=2`.

</details>

<details>
<summary>💡 Hint 2</summary>

The win for one instrument is `100 * (line_err[t] - tree_err[t]) / line_err[t]`. Sort the winners with `sorted(..., key=..., reverse=True)`.

</details>

<details>
<summary>✅ Solution</summary>

```python
tree_err = {}
line_err = {}
for ticker in tickers:
    frame = vol_table(ticker)
    early, late = frame.loc[:'2022-12-31'], frame.loc['2023-01-01':]
    t = DecisionTreeRegressor(max_depth=2, random_state=0).fit(early[pair], early['vol_next'])
    l = LinearRegression().fit(early[pair], early['vol_next'])
    tree_err[ticker] = round(float(rmse(late['vol_next'], t.predict(late[pair]))), 4)
    line_err[ticker] = round(float(rmse(late['vol_next'], l.predict(late[pair]))), 4)

wins = sorted([t for t in tickers if tree_err[t] < line_err[t]],
              key=lambda t: (line_err[t] - tree_err[t]) / line_err[t], reverse=True)
for ticker in wins:
    print(ticker, round(100 * (line_err[ticker] - tree_err[ticker]) / line_err[ticker], 1))
```

The tree wins on 3 of the 11: JNJ by 1.0 percent, JPM by 0.7 percent and AAPL by 0.4 percent. The wins are small. Four leaves rarely beat a slope on a target that moves as smoothly as volatility, even with the direction of the market to ask about.

</details>

---

### J5 · Draw the desk  ★★☆☆☆  · revisits S3

Using `stump_err` and `line_err` from J2, or rebuilding them, draw the ratio of the stump's RMSE to the line's for every instrument as bars, with a dashed line at 1. A bar below the line is a win for the stump.

In [ ]:
stump_err = {}
line_err = {}
for ticker in tickers:
    ...

ratio = ...

fig, ax = plt.subplots(figsize=(8, 3))
...
...
ax.set_ylabel('stump RMSE / line RMSE')
plt.show()

<details>
<summary>💡 Hint 1</summary>

Rebuild the two dictionaries with the loop from J2, then `ratio = pd.Series({t: stump_err[t] / line_err[t] for t in tickers})`.

</details>

<details>
<summary>💡 Hint 2</summary>

`ax.bar(ratio.index, ratio.values)` and `ax.axhline(1, linestyle='--', color='grey')`.

</details>

<details>
<summary>✅ Solution</summary>

```python
stump_err = {}
line_err = {}
for ticker in tickers:
    frame = vol_table(ticker)
    early, late = frame.loc[:'2022-12-31'], frame.loc['2023-01-01':]
    s = DecisionTreeRegressor(max_depth=1, random_state=0).fit(early[['vol_20d']], early['vol_next'])
    l = LinearRegression().fit(early[['vol_20d']], early['vol_next'])
    stump_err[ticker] = rmse(late['vol_next'], s.predict(late[['vol_20d']]))
    line_err[ticker] = rmse(late['vol_next'], l.predict(late[['vol_20d']]))

ratio = pd.Series({t: stump_err[t] / line_err[t] for t in tickers})

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(ratio.index, ratio.values, color='#1c5cab')
ax.axhline(1, color='grey', linestyle='--')
ax.set_ylabel('stump RMSE / line RMSE')
plt.show()
```

Most bars stand above 1, and XOM furthest, at 1.541. A ratio puts eleven instruments with very different volatility on one scale, which the raw RMSEs cannot do.

</details>

---

---

## K · Five small cases

Each of these stands completely on its own and needs no model from earlier in the notebook. They are here to keep loops, `while`, dictionaries, f-strings and functions in working order, in this session's setting.

### K1 · A tree as if-statements  ★★☆☆☆  · revisits S2

The lecture's tree of depth 2 is written out below as rules. Write `forecast(vol, ret)` that follows them, and print the forecast for each of the four days in `days`.

- `vol` 0.99 or less: forecast 0.70 if `vol` is 0.82 or less, otherwise 0.99
- `vol` above 0.99: forecast 2.84 if `ret` is -0.45 or less, otherwise 1.25

In [ ]:
def forecast(vol, ret):
    """..."""
    ...


days = [(0.55, 0.10), (0.90, -0.20), (1.40, -0.70), (2.10, 0.05)]
for vol, ret in days:
    ...

<details>
<summary>💡 Hint 1</summary>

An outer `if vol <= 0.99:` and an `else:`, each with its own `if`.

</details>

<details>
<summary>💡 Hint 2</summary>

In the loop: `print(vol, ret, forecast(vol, ret))`.

</details>

<details>
<summary>✅ Solution</summary>

```python
def forecast(vol, ret):
    """The depth-2 tree of the lecture, as nested if-statements."""
    if vol <= 0.99:
        if vol <= 0.82:
            return 0.70
        return 0.99
    if ret <= -0.45:
        return 2.84
    return 1.25


days = [(0.55, 0.10), (0.90, -0.20), (1.40, -0.70), (2.10, 0.05)]
for vol, ret in days:
    print(vol, ret, forecast(vol, ret))
```

The four forecasts are [0.7, 0.99, 2.84, 1.25]. A fitted tree is nothing more than this: nested `if` statements whose cuts and numbers were chosen from the data.

</details>

---

### K2 · A tree's questions, as text  ★★☆☆☆  · revisits S2

Inside a fitted tree, `tree_.feature` holds the column number of each box's question, with -2 for a leaf, and `tree_.threshold` holds its cut. Below are the two arrays of a tree of depth 2 fitted on three columns of the index table, with the cuts rounded to three decimals. Loop over the boxes with `zip` and, for every box that asks a question, add the text of the question, such as `'vol_20d <= 0.841'`, to `questions` with an f-string. Then count the different columns the tree asks about with a set.

In [ ]:
names = ['vol_20d', 'ret_20d', 'vol_5d']
feature = [2, 0, -2, -2, 0, -2, -2]
threshold = [3.251, 0.841, -2.0, -2.0, 4.587, -2.0, -2.0]

questions = []
for f, t in zip(feature, threshold):
    ...

asked = ...

print(questions)
print(len(questions), 'questions about', ..., 'of', len(names), 'columns')

<details>
<summary>💡 Hint 1</summary>

A box asks a question when `f >= 0`, and `names[f]` is its column: `questions.append(f'{names[f]} <= {t}')`.

</details>

<details>
<summary>💡 Hint 2</summary>

`set(names[f] for f in feature if f >= 0)` keeps each column once, and `len(asked)` counts them.

</details>

<details>
<summary>✅ Solution</summary>

```python
names = ['vol_20d', 'ret_20d', 'vol_5d']
feature = [2, 0, -2, -2, 0, -2, -2]
threshold = [3.251, 0.841, -2.0, -2.0, 4.587, -2.0, -2.0]

questions = []
for f, t in zip(feature, threshold):
    if f >= 0:
        questions.append(f'{names[f]} <= {t}')

asked = set(names[f] for f in feature if f >= 0)

print(questions)
print(len(questions), 'questions about', len(asked), 'of', len(names), 'columns')
```

`['vol_5d <= 3.251', 'vol_20d <= 0.841', 'vol_20d <= 4.587']`: three questions about two of the three columns. The first asks about `vol_5d`, and its cut at 3.251 percent sets apart 27 of the 1,894 training days, 26 of them in March and April 2020; both boxes below it then ask about `vol_20d`. Offered `ret_20d`, this tree never used it. The boxes are numbered depth first, which is why the second question sits at position 1 and the third at position 4.

</details>

---

### K3 · Until every day has been left out once  ★★★★☆  · revisits S2

A day gets an out-of-bag forecast only once some tree's sample has left it out. Draw bootstrap samples of the training days one at a time, with `random_state` 1, 2, 3 and so on, and after each one mark the days it left out. Stop with a `break` as soon as every day has been left out at least once, and print how many samples that took. The `while` is bounded at 100.

In [ ]:
left_out_once = np.zeros(len(train), dtype=bool)
samples = 0
while samples < 100:
    samples += 1
    ...

print(samples)

<details>
<summary>💡 Hint 1</summary>

`~train.index.isin(sample.index)` is True for the days one sample left out, and `left_out_once | ...` adds them to the days left out before.

</details>

<details>
<summary>💡 Hint 2</summary>

`left_out_once.all()` is True once every day has been left out; `break` there.

</details>

<details>
<summary>✅ Solution</summary>

```python
left_out_once = np.zeros(len(train), dtype=bool)
samples = 0
while samples < 100:
    samples += 1
    sample = train.sample(n=len(train), replace=True, random_state=samples)
    left_out_once = left_out_once | ~train.index.isin(sample.index)
    if left_out_once.all():
        break

print(samples)
```

17 samples. After 10 there were still 33 days that every sample had drawn. A day is left out of one sample with a chance of about 0.368, so it is in all of k samples with a chance of 0.632 to the power k, and the expected number of days never left out, 1,894 times that, falls below one at k = 17. A forest of 100 trees gives every day plenty of trees that never saw it.

</details>

---

### K4 · The best cell of a grid, by hand  ★★★☆☆  · revisits S2

Below are the mean fold RMSEs of a grid of forests: one row per `max_features`, one column per `min_samples_leaf`. Find the best cell with two nested loops over the positions, keeping the best score so far and its pair of settings. Then find the same cell with NumPy: `argmin` on the array gives one position, counted along the first row, then the second, and `//` and `%` by the number of columns turn it into a row and a column.

In [ ]:
max_features = [1, 4, 19]
leaf_sizes = [5, 20, 50, 100, 200]
scores = [[0.639, 0.620, 0.593, 0.601, 0.615],
          [0.715, 0.662, 0.620, 0.613, 0.628],
          [0.804, 0.741, 0.696, 0.682, 0.705]]

best_score = scores[0][0]
best_pair = (max_features[0], leaf_sizes[0])
for i in range(len(max_features)):
    for j in range(len(leaf_sizes)):
        ...

print(best_pair, best_score)

grid = np.array(scores)
position = ...
row = ...
col = ...
same = ...

print(position, row, col)
print(same)

<details>
<summary>💡 Hint 1</summary>

Inside the inner loop: `if scores[i][j] < best_score:`, then update both `best_score` and `best_pair = (max_features[i], leaf_sizes[j])`.

</details>

<details>
<summary>💡 Hint 2</summary>

`position = int(grid.argmin())`, `row = position // grid.shape[1]` and `col = position % grid.shape[1]`. `same` compares `(max_features[row], leaf_sizes[col])` with `best_pair`.

</details>

<details>
<summary>✅ Solution</summary>

```python
max_features = [1, 4, 19]
leaf_sizes = [5, 20, 50, 100, 200]
scores = [[0.639, 0.620, 0.593, 0.601, 0.615],
          [0.715, 0.662, 0.620, 0.613, 0.628],
          [0.804, 0.741, 0.696, 0.682, 0.705]]

best_score = scores[0][0]
best_pair = (max_features[0], leaf_sizes[0])
for i in range(len(max_features)):
    for j in range(len(leaf_sizes)):
        if scores[i][j] < best_score:
            best_score = scores[i][j]
            best_pair = (max_features[i], leaf_sizes[j])

print(best_pair, best_score)

grid = np.array(scores)
position = int(grid.argmin())
row = position // grid.shape[1]
col = position % grid.shape[1]
same = (max_features[row], leaf_sizes[col]) == best_pair

print(position, row, col)
print(same)
```

`(1, 50)` at 0.593, and `True`: position 2 is row 0, column 2. These are the fold scores of G3's grid, rounded, and the two loops do what `GridSearchCV` does once every cell is scored: keep the best and store its settings in `best_params_`. `//` counts the whole rows passed and `%` what is left over, the same arithmetic as turning minutes into hours and minutes.

</details>

---

### K5 · Leaf forecasts, by hand  ★★★☆☆  · revisits S2

Below are ten training days of the index table, one every 180 trading days: the leaf of the lecture's tree of depth 2 each one landed in, and what `vol_next` turned out to be. Collect the `vol_next` values of each leaf in a dictionary of lists with a loop, then build `means`, the mean of each leaf to three decimals. Check the means with pandas `groupby` in one line.

In [ ]:
landed = [2, 3, 2, 2, 2, 6, 2, 6, 6, 3]
vol_next = [0.85, 0.63, 0.46, 0.57, 0.64, 0.65, 0.45, 0.86, 0.73, 1.11]

by_leaf = {}
for leaf, value in zip(landed, vol_next):
    ...

means = {}
for leaf in by_leaf:
    ...

check = ...

print(means)
print(check)

<details>
<summary>💡 Hint 1</summary>

Inside the first loop: `if leaf not in by_leaf:` start an empty list, then `by_leaf[leaf].append(value)`.

</details>

<details>
<summary>💡 Hint 2</summary>

`means[leaf] = round(sum(by_leaf[leaf]) / len(by_leaf[leaf]), 3)`, and `pd.Series(vol_next).groupby(landed).mean().round(3)` is the one line.

</details>

<details>
<summary>✅ Solution</summary>

```python
landed = [2, 3, 2, 2, 2, 6, 2, 6, 6, 3]
vol_next = [0.85, 0.63, 0.46, 0.57, 0.64, 0.65, 0.45, 0.86, 0.73, 1.11]

by_leaf = {}
for leaf, value in zip(landed, vol_next):
    if leaf not in by_leaf:
        by_leaf[leaf] = []
    by_leaf[leaf].append(value)

means = {}
for leaf in by_leaf:
    means[leaf] = round(sum(by_leaf[leaf]) / len(by_leaf[leaf]), 3)

check = pd.Series(vol_next).groupby(landed).mean().round(3)

print(means)
print(check)
```

`{2: 0.594, 3: 0.87, 6: 0.747}`, and `groupby` gives the same three numbers. The tree's own forecasts for these leaves are the same calculation on all their training days: 0.698 from 969 days, 0.994 from 185 and 1.252 from 693. Ten days give only a rough version of them, which is why a leaf needs many days before its mean can be trusted.

</details>

---

## 🏁 Done

You found the best number for a box and the best cut with loops and NumPy, checked both against scikit-learn, drew trees and traced days through them, watched a deep tree fit the training days and miss the test days, chose the size of a tree on the time-series folds, drew bootstrap samples and bagged trees by hand, counted what `max_features` changes, found out-of-bag scores too kind on days and fair on borrowers, counted, timed and sampled grids, and fixed five mistakes that trees make easy.

The case gives the risk report's volatility forecast a tree and a forest, held to its one-column model and to ridge, on Apple and across the desk, and ends with one question on Part 9's jump label.